# Đọc file text trên HDFS bằng Spark

Đọc các file mà pipeline NiFi **Local-to-HDFS** (`GetFile` → `PutHDFS`) đã đẩy lên thư mục `/user/nifi/incoming` trên HDFS.

Thả file `.txt` vào `streaming/nifi/data/` trên host → NiFi tự động đưa lên HDFS trong ~10 giây → chạy notebook này để đọc lại.

## 1. Tạo SparkSession

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("read-hdfs-text")
    .master("spark://spark-master:7077")   # đổi thành local[*] để chạy local, không cần worker
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/10 02:26:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 2. Đọc tất cả file .txt trong thư mục thành 1 DataFrame

`fs.defaultFS` đã trỏ sẵn về `hdfs://hadoop-namenode:9000`, nên chỉ cần dùng đường dẫn tương đối `/user/...` là đọc thẳng trên HDFS (không cần ghi `hdfs://` đầy đủ).

In [2]:
HDFS_DIR = "/user/nifi/incoming"

# Mỗi dòng trong mỗi file -> 1 row trong cột "value"
df = spark.read.text(HDFS_DIR)
print("số dòng:", df.count())
df.show(n=50, truncate=False)

số dòng: 85
+-------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                            |
+-------------------------------------------------------------------------------------------------------------------------------------------------+
| Hãy triển khai chức năng “Phân tích ảnh hưởng theo FQN” khi tôi chọn 1 fqn tại 1 file code mơ hồ theo đúng luồng hiện có của agent-api.         |
|                                                                                                                                                 |
|  Mục tiêu:                                                                                                                                      |
|  Khi người dùng chọn một entity FQN trên UI và yêu cầu phân tích ảnh hưởng, frontend phải gửi FQN 

+-------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                            |
+-------------------------------------------------------------------------------------------------------------------------------------------------+
| Hãy triển khai chức năng “Phân tích ảnh hưởng theo FQN” khi tôi chọn 1 fqn tại 1 file code mơ hồ theo đúng luồng hiện có của agent-api.         |
|                                                                                                                                                 |
|  Mục tiêu:                                                                                                                                      |
|  Khi người dùng chọn một entity FQN trên UI và yêu cầu phân tích ảnh hưởng, frontend phải gửi FQN đó qua `page

## 3. Biết mỗi dòng thuộc file nào

Dùng `input_file_name()` để lấy đường dẫn file gốc kèm theo từng dòng — hữu ích khi thư mục có nhiều file trộn lẫn.

In [3]:
from pyspark.sql import functions as F

df_with_source = df.withColumn("source_file", F.input_file_name())
df_with_source.select("source_file", "value").show(n=50, truncate=False)

+--------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------+
|source_file                                                                                       |value                                                                                                                                            |
+--------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------+
|hdfs://hadoop-namenode:9000/user/nifi/incoming/Hãy%20triển%20khai%20chức%20năng%20“Phân%20tích.txt| Hãy triển khai chức năng “Phân tích ảnh hưởng theo FQN” khi tôi chọn 1 fqn tại 1 file code mơ hồ theo đúng luồng hiện có của agent-api.         |
|hdfs://hado

## 4. Đọc nguyên nội dung từng file (không tách dòng)

`wholeTextFiles` trả về RDD các cặp `(đường_dẫn_file, toàn_bộ_nội_dung_file)` — mỗi file là 1 phần tử, phù hợp khi cần xử lý theo file thay vì theo dòng.

In [4]:
files_rdd = spark.sparkContext.wholeTextFiles(f"{HDFS_DIR}/*.txt")

for path, content in files_rdd.collect():
    print("=== ", path, " ===")
    print(content[:300])
    print()

===  hdfs://hadoop-namenode:9000/user/nifi/incoming/Hãy triển khai chức năng “Phân tích.txt  ===
 Hãy triển khai chức năng “Phân tích ảnh hưởng theo FQN” khi tôi chọn 1 fqn tại 1 file code mơ hồ theo đúng luồng hiện có của agent-api.

  Mục tiêu:
  Khi người dùng chọn một entity FQN trên UI và yêu cầu phân tích ảnh hưởng, frontend phải gửi FQN đó qua `page_context.entity_fqn` tới API chat.


===  hdfs://hadoop-namenode:9000/user/nifi/incoming/entry_points.txt  ===
[console_scripts]
watchfiles=watchfiles.cli:cli


===  hdfs://hadoop-namenode:9000/user/nifi/incoming/test-1789005290.txt  ===
hello from local Thu Sep 10 08:54:50 SEAST 2026




Kiểm tra danh sách file bằng terminal:
```bash
docker exec hadoop-namenode hdfs dfs -ls /user/nifi/incoming
```
hoặc mở http://localhost:9870 → Utilities → Browse the file system.

## 5. Dọn dẹp

In [5]:
spark.stop()